<a href="https://colab.research.google.com/github/MariamMohamed06/flyrank-ml-week1/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MariamMohamed06/flyrank-ml-week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

The warehouse fact table has one row per content page, client, and day.

For my lane, the decision is made at the content-page level by using signals observed during a defined time window.

I will start with March 2026 as the feature window because it is a middle month in the available history and helps avoid using the final month as development data.

The goal is to rank content pages for review based on evidence available during this window.

In [ ]:
import os
import duckdb
from google.colab import userdata

# Get the token from Colab Secrets
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Let DuckDB use the Hugging Face token from the credential chain
con.execute("""
CREATE SECRET hf_token (
    TYPE HUGGINGFACE,
    PROVIDER credential_chain
)
""")

# Warehouse location
warehouse = "hf://datasets/FlyRank/internship-warehouse"

## 2. Fields: feature / label / context / excluded
### Fields

#### Features
- `impressions`: search visibility measured before the decision.
- `clicks`: search clicks measured before the decision.
- `sessions`: traffic measured before the decision.
- `avg_position`: average search position measured before the decision.
- `content_age_days`: page age known at the decision moment.

#### Label / Proxy
- `trend_direction`: used as a starter proxy for whether the page is declining.
- `is_declining_label`: `1` when `trend_direction == "down"`, otherwise `0`.

#### Context
- `client_hash_id`: identifies the client for grouping and validation, but is not used as a predictive feature.
- `content_hash_id`: identifies the content page and is used to define the unit of analysis.

#### Excluded
- Raw or sensitive identifiers, URLs, client names, queries, and titles are excluded because the released data is designed to be public-safe.
- Any future-window measurements are excluded from features because they could leak information from the outcome period.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
query1 = """
SELECT
    COUNT(*) AS rows_count,
    COUNT(DISTINCT content_hash_id || '|' || client_hash_id || '|' || CAST(report_date AS VARCHAR)) AS unique_grain_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query1)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬───────────────────┐
│ rows_count │ unique_grain_rows │
│   int64    │       int64       │
├────────────┼───────────────────┤
│    9841378 │           9841378 │
└────────────┴───────────────────┘

In [ ]:
query2 = """
SELECT
    COUNT(*) AS rows_count,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query2)

┌────────────┬────────────┬────────────┐
│ rows_count │ first_date │ last_date  │
│   int64    │    date    │    date    │
├────────────┼────────────┼────────────┤
│    9841378 │ 2026-03-01 │ 2026-03-31 │
└────────────┴────────────┴────────────┘

In [ ]:
query3 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

con.sql(query3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘

In [ ]:
columns = con.sql("""
DESCRIBE
SELECT *
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
""")

columns

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

In [ ]:
con.sql("""
SELECT column_name
FROM (
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
)
""")

┌────────────────────┐
│    column_name     │
│      varchar       │
├────────────────────┤
│ report_date        │
│ client_hash_id     │
│ content_hash_id    │
│ client_has_gsc     │
│ client_has_ga4     │
│ gsc_data_available │
│ ga4_data_available │
│ gsc_impressions    │
│ gsc_clicks         │
│ gsc_sum_position   │
│      ·             │
│      ·             │
│      ·             │
│ sessions_ai        │
│ ai_chatgpt         │
│ ai_perplexity      │
│ ai_gemini          │
│ ai_copilot         │
│ ai_claude          │
│ ai_meta            │
│ ai_other           │
│ scroll_events      │
│ month              │
├────────────────────┤
│ 31 rows (20 shown) │
└────────────────────┘

In [ ]:
all_columns = con.sql("""
SELECT column_name
FROM (
    DESCRIBE
    SELECT *
    FROM read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    )
)
""").fetchall()

for i, col in enumerate(all_columns, 1):
    print(i, col[0])

1 report_date
2 client_hash_id
3 content_hash_id
4 client_has_gsc
5 client_has_ga4
6 gsc_data_available
7 ga4_data_available
8 gsc_impressions
9 gsc_clicks
10 gsc_sum_position
11 gsc_avg_position
12 ga4_pageviews
13 ga4_sessions
14 ga4_users
15 ga4_engaged_sessions
16 ga4_total_engagement_sec
17 sessions_organic
18 sessions_direct
19 sessions_referral
20 sessions_social
21 sessions_paid
22 sessions_ai
23 ai_chatgpt
24 ai_perplexity
25 ai_gemini
26 ai_copilot
27 ai_claude
28 ai_meta
29 ai_other
30 scroll_events
31 month


### Five features

I will use five features from the March 2026 feature window:

1. `gsc_impressions` — available at the decision moment because search impressions have already been observed during the feature window.

2. `gsc_clicks` — available at the decision moment because search clicks have already been observed during the feature window.

3. `gsc_avg_position` — available at the decision moment because average search position is calculated from the observed search data.

4. `ga4_sessions` — available at the decision moment when GA4 data is available for the page.

5. `scroll_events` — available at the decision moment when the observed engagement data is available.

These features are used to describe the page's observed search visibility, traffic, and engagement before making a review-prioritization decision.

In [ ]:
features = con.sql("""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(scroll_events) AS scroll_events
FROM read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,scroll_events
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.147402,NaN,NaN
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,4.828125,NaN,NaN
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.145765,NaN,NaN
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,4.909314,NaN,NaN
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,NaN,NaN


In [ ]:
# Deliberate leakage experiment:
# Create a label from the same March outcome window.
leak_test = features.copy()

leak_test["went_dark"] = (leak_test["gsc_clicks"].fillna(0) == 0).astype(int)

# Deliberately leak the label itself as a "score"
leak_test["leaky_score"] = leak_test["went_dark"]

k = 50

top_50 = leak_test.nlargest(k, "leaky_score")

precision_at_50_leaky = top_50["went_dark"].mean()

print(f"Leaky Precision@50: {precision_at_50_leaky:.3f}")

Leaky Precision@50: 1.000


In [ ]:
# Remove the intentionally leaked column.
features_honest = features.drop(columns=["leaky_score"], errors="ignore")

print("Honest feature columns:")
print(features_honest.columns.tolist())

Honest feature columns:
['client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_sessions', 'scroll_events']


## 4. Data limits

A key limitation is uneven data availability across pages and clients. In the March 2026 slice, GSC and GA4 signals are not available for every row, so feature coverage is incomplete.

The March slice is also only one month, so it may not represent longer-term behavior or seasonal patterns.

Finally, the starter proxy is based on observed data rather than a true future outcome, so it should be treated as decision-support evidence rather than causal proof.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.